In [4]:
import pandas as pd

df = pd.read_csv("health_data.csv")
print(df.head())

   Patient_id   Patient_Name  Age  Gender Medical Condition Date of Admission  \
0         328  Bobby Jackson   30    Male            Cancer        31-01-2024   
1         265   Leslie Terry   62    Male           Obesity        20-08-2019   
2         205    Danny Smith   76  Female           Obesity        22-09-2022   
3         450   Andrew Watts   28  Female          Diabetes        18-11-2020   
4         458  Adrienne Bell   43  Female            Cancer        19-09-2022   

   Billing Amount  
0     18856.28131  
1     33643.32729  
2     27955.09608  
3     37909.78241  
4     14238.31781  


This is where we have loaded the dataset.

In [5]:
print(df.columns)
print(df.info())
print(df.describe())

Index(['Patient_id', 'Patient_Name', 'Age', 'Gender', 'Medical Condition',
       'Date of Admission', 'Billing Amount'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 999 entries, 0 to 998
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Patient_id         999 non-null    int64  
 1   Patient_Name       999 non-null    object 
 2   Age                999 non-null    int64  
 3   Gender             999 non-null    object 
 4   Medical Condition  999 non-null    object 
 5   Date of Admission  999 non-null    object 
 6   Billing Amount     999 non-null    float64
dtypes: float64(1), int64(2), object(4)
memory usage: 54.8+ KB
None
       Patient_id         Age  Billing Amount
count  999.000000  999.000000      999.000000
mean   295.170170   50.972973    25099.622670
std    116.942534   19.687241    14460.387378
min    101.000000   18.000000    -1018.245371
25%    192.500000   34

This is where we check the structure and information like columns and datatypes of the dataset.

In [9]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

Standardizing the names are very important for further development of the project.so, that is what happens here like removing spaces coverting in to lowercase.

In [10]:
df['date_of_admission'] = pd.to_datetime(df['date_of_admission'])

/tmp/ipykernel_11191/3185648414.py:1: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['date_of_admission'] = pd.to_datetime(df['date_of_admission'])


Changed the name of the column for better development.

In [11]:
import numpy as np

df['date_of_discharge'] = df['date_of_admission'] + pd.to_timedelta(
    np.random.randint(1, 15, size=len(df)), unit='D'
)

Adding date of discharge column to our dataset

In [13]:
df['length_of_stay'] = (df['date_of_discharge'] - df['date_of_admission']).dt.days

Calculates the number of days patients stayed.

In [14]:
df = df.sort_values(by=['patient_id', 'date_of_admission'])

This is where it sorts and arranges the visits of the patients.

In [15]:
df['days_between_visits'] = df.groupby('patient_id')['date_of_admission'].diff().dt.days

Calculates the difference between the visits of each patient.

In [16]:
df['readmission_flag'] = df['days_between_visits'].apply(
    lambda x: 1 if pd.notnull(x) and x <= 30 else 0
)

We have taken 30 days as base and if the patient visits after 30 days we take that as a readmission.

In [17]:
df['age_group'] = pd.cut(df['age'], bins=[0,18,60,100], labels=['Child','Adult','Senior'])

This is where we create age groups for better analysis.

In [18]:
df.to_csv("clean_health_data.csv", index=False)

After all the analysis this is the final dataset.

In [26]:
import pandas as pd

df = pd.read_csv("clean_health_data.csv")

In [28]:
# Features
X = df[['age', 'billing_amount', 'length_of_stay']]

# Add categorical columns
X = pd.get_dummies(
    X.join(df[['gender','medical_condition']]),
    drop_first=True
)

# Target
y = df['readmission_flag']

In [29]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [30]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()
model.fit(X_train, y_train)

RandomForestClassifier()

In [31]:
y_pred = model.predict(X_test)

In [32]:
from sklearn.metrics import accuracy_score

print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.945
